In [1]:
import joblib
import re
import numpy as np
import xgboost as xgb
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import LabelEncoder

In [2]:
file_path = "./Tello_comand_dataset_train.xlsx"
df = pd.read_excel(file_path)

In [3]:
df.head()
df.tail()

,Human Command,Command
115,How's the battery looking?,battery?
116,Tell me the battery percentage,battery?
117,Give me the current battery level,battery?
118,How much battery is remaining?,battery?
119,What's the current battery amount?,battery?


In [4]:
X = []
y = []

In [5]:
for sentence, command in zip(df['Human Command'], df['Command']):
    # Simple slot-filling extraction for 'x' values (look for numeric values)
    x_value = re.findall(r"\d+", sentence)
    x_value = x_value[0] if x_value else None
    X.append([sentence, x_value])
    y.append(command)  # Labels are already clean

In [6]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.3, random_state=42)

In [8]:
def extract_features(text, x_value):
    # Extract basic features: Length, token count, and presence of x value
    features = [
        len(text),                      # Length of the sentence
        len(text.split()),               # Number of tokens
        1 if bool(x_value) else 0,       # Whether there is an 'x' value (slot)
    ]
    
    # Extract keyword-based features (presence of important words)
    keywords = ["command", "takeoff", "land", "streamon", "streamoff", "up x", "down x", "left x", "right x"
                "forward x", "back x", "battery?"]
    for keyword in keywords:
        features.append(int(keyword in text.lower()))  # Presence of keyword
    
    return features

In [9]:
train_features = [extract_features(text, x) for text, x in X_train]
test_features = [extract_features(text, x) for text, x in X_test]

In [10]:
classes = np.array(list(set(y_train)))  # Convert classes to numpy array
class_weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, class_weights))


In [11]:
dtrain = xgb.DMatrix(train_features, label=y_train)
dtest = xgb.DMatrix(test_features, label=y_test)

In [12]:
params = {
    'objective': 'multi:softmax',  # For multi-class classification
    'num_class': len(classes),     # Number of unique classes
    'eval_metric': 'merror',       # Evaluation metric: misclassification error
    'max_depth': 6,                # Depth of the tree
    'eta': 0.3,                    # Learning rate
    'silent': 1                    # Suppress warnings
}

In [13]:
bst = xgb.train(params, dtrain, num_boost_round=100)

/Users/biggy/big/internship/RAG/internship-rag/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [16:25:11] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:738: 
Parameters: { "silent" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [14]:
y_pred = bst.predict(dtest)
print(classification_report(y_test, y_pred.astype(int)))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00         3
           1       0.33      0.50      0.40         2
           2       0.40      1.00      0.57         2
           3       0.00      0.00      0.00         3
           4       0.00      0.00      0.00         3
           5       1.00      0.67      0.80         3
           6       1.00      0.25      0.40         4
           7       0.00      0.00      0.00         2
           8       1.00      0.20      0.33         5
           9       0.00      0.00      0.00         2
          10       0.00      0.00      0.00         5
          11       0.00      0.00      0.00         2

    accuracy                           0.19        36
   macro avg       0.31      0.22      0.21        36
weighted avg       0.37      0.19      0.21        36



/Users/biggy/big/internship/RAG/internship-rag/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/biggy/big/internship/RAG/internship-rag/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/biggy/big/internship/RAG/internship-rag/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(av

In [15]:
joblib.dump(bst, 'tello_command_model_xg.pkl')

['tello_command_model_xg.pkl']